In [ ]:
!nvidia-smi

Tue Jun  2 13:15:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!git clone https://github.com/Kush-Singh-26/ember.git
%cd ember


Cloning into 'ember'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 64 (delta 18), reused 61 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (64/64), 1.02 MiB | 3.06 MiB/s, done.
Resolving deltas: 100% (18/18), done.
/content/ember


In [ ]:
import torch
import os
from safetensors.torch import load_file
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from model.transformer import EmberForCausalLM

# ==========================================
# 1. SETUP & MODEL LOADING
# ==========================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Download weights
checkpoint_dir = snapshot_download(
    repo_id="Kush26/ember-checkpoints",
    revision="checkpoints",
    allow_patterns="checkpoints/ember-275m/checkpoint-3650/*"
)
model_path = os.path.join(checkpoint_dir, "checkpoints/ember-275m/checkpoint-3650")

# Load model using the verified manual loading method
config = EmberForCausalLM.config_class.from_pretrained(model_path)
model = EmberForCausalLM(config)
state_dict = load_file(os.path.join(model_path, "model.safetensors"))
state_dict["lm_head.weight"] = state_dict["model.embed_tokens.weight"].clone()
model.load_state_dict(state_dict, strict=True)
model = model.to(device)
model.eval()

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained("Kush26/ember-tokenizer")

# ==========================================
# 2. ROBUST INFERENCE FUNCTION
# ==========================================
def generate_text(
    prompt: str,
    max_new_tokens: int = 120,
    do_sample: bool = False,
    temperature: float = 0.2,
    top_p: float = 0.95,
) -> str:
    """
    Generates continuation text for a given prompt using the Ember model.

    Args:
        prompt: The input code/text string.
        max_new_tokens: Number of tokens to generate.
        do_sample: If True, uses nucleus sampling. If False, uses greedy decoding.
        temperature: Sampling temperature (higher is more creative/random).
        top_p: Nucleus sampling cumulative probability threshold.
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # Configure generation parameters
    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "pad_token_id": tokenizer.pad_token_id,
        "bos_token_id": tokenizer.bos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }

    if do_sample:
        gen_kwargs["temperature"] = temperature
        gen_kwargs["top_p"] = top_p

    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_kwargs)

    # Decode the output, omitting special tokens for clean prints
    generated_sequence = outputs[0]
    return tokenizer.decode(generated_sequence, skip_special_tokens=True)


# ==========================================
# 3. LIST OF DIVERSE PROMPTS TO RUN
# ==========================================
# prompts_to_test = [
#     # 1. Classic Algorithm Completion
#     "def quicksort(arr):",

#     # 2. String Manipulation with Docstring
#     "def is_palindrome(s: str) -> bool:\n    \"\"\"Check if a string is a palindrome.\"\"\"",

#     # 3. Web Programming (JS / Fetch API)
#     "// JavaScript function to fetch user data from API\nasync function fetchUserData(userId) {",

#     # 4. Standard Math / Recursion
#     "def fibonacci(n):",

#     # 5. Database querying (SQL)
#     "-- SQL Query to select all users older than 21 in 'users' table\nSELECT",
# ]



Using device: cuda


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]


=== STARTING INFERENCE TESTS ===

----------------------------------------
TEST 1: PROMPT:
how are you?
----------------------------------------
GENERATION:
how are you? If you want to use a
            string, you can use a string, or a string.

        :param str name: The name of the value to be used for the value.
        :param str value: The value to be used for the value.
        :param str value_type: The type of the value.
        :param str value_type_name: The type of the

----------------------------------------
TEST 2: PROMPT:
who is newton
----------------------------------------
GENERATION:
who is newtonic and the only other major league in the world . 



In [ ]:
prompts_to_test = [
    "how are you?",
    "who is newton",
]
# Run the prompts through the inference engine
print("\n=== STARTING INFERENCE TESTS ===")
for i, prompt in enumerate(prompts_to_test, 1):
    print(f"\n----------------------------------------")
    print(f"TEST {i}: PROMPT:\n{prompt}")
    print(f"----------------------------------------")

    # Using deterministic greedy decoding (ideal for standard code completions)
    completion = generate_text(prompt, max_new_tokens=80, do_sample=False)

    print(f"GENERATION:\n{completion}")


=== STARTING INFERENCE TESTS ===

----------------------------------------
TEST 1: PROMPT:
how are you?
----------------------------------------
GENERATION:
how are you? If you want to use a
            string, you can use a string, or a string.

        :param str name: The name of the value to be used for the value.
        :param str value: The value to be used for the value.
        :param str value_type: The type of the value.
        :param str value_type_name: The type of the

----------------------------------------
TEST 2: PROMPT:
who is newton
----------------------------------------
GENERATION:
who is newtonic and the only other major league in the world . 



In [ ]:
import torch
import os
from safetensors.torch import load_file
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from model.transformer import EmberForCausalLM

# ==========================================
# 1. SETUP & MODEL LOADING
# ==========================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Download weights
checkpoint_dir = snapshot_download(
    repo_id="Kush26/ember-checkpoints",
    revision="checkpoints",
    allow_patterns="checkpoints/ember-275m/checkpoint-3650/*"
)
model_path = os.path.join(checkpoint_dir, "checkpoints/ember-275m/checkpoint-3650")

# Load model via manually-tied state dict (prevents Hugging Face tying overwrite)
config = EmberForCausalLM.config_class.from_pretrained(model_path)
model = EmberForCausalLM(config)
state_dict = load_file(os.path.join(model_path, "model.safetensors"))
state_dict["lm_head.weight"] = state_dict["model.embed_tokens.weight"].clone()
model.load_state_dict(state_dict, strict=True)
model = model.to(device)
model.eval()

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained("Kush26/ember-tokenizer")

# ==========================================
# 2. DEFINING THE INFERENCE FUNCTION
# ==========================================
def generate_text(
    prompt: str,
    max_new_tokens: int = 80,
    do_sample: bool = True,           # Enable sampling by default for diversity
    temperature: float = 0.7,         # Standard temperature
    top_p: float = 0.9,               # Nucleus sampling threshold
    repetition_penalty: float = 1.2,  # CRITICAL: Breaks deterministic repetition loops
) -> str:
    """
    Generates text/code for a prompt using the Ember model with repetition penalties.
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "repetition_penalty": repetition_penalty,
        "pad_token_id": tokenizer.pad_token_id,
        "bos_token_id": tokenizer.bos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
        # Restrict repetitions on a 3-gram level for safety
        "no_repeat_ngram_size": 3,
    }

    if do_sample:
        gen_kwargs["temperature"] = temperature
        gen_kwargs["top_p"] = top_p

    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_kwargs)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# ==========================================
# 3. RUNNING INFERENCE ON PROMPTS
# ==========================================
prompts_to_test = [
    "def quicksort(arr):",
    "def is_palindrome(s: str) -> bool:\n    \"\"\"Check if a string is a palindrome.\"\"\"",
    "def fibonacci(n):",
    "// JavaScript function to fetch user data from API\nasync function fetchUserData(userId) {",
    "-- SQL Query to select all users older than 21 in 'users' table\nSELECT"
]

print("\n=== STARTING INFERENCE TESTS ===")
for i, prompt in enumerate(prompts_to_test, 1):
    print(f"\n----------------------------------------")
    print(f"TEST {i}: PROMPT:\n{prompt}")
    print(f"----------------------------------------")

    # Run with default Nucleus Sampling + Repetition Penalty
    completion = generate_text(prompt, max_new_tokens=60, do_sample=True)

    print(f"GENERATION:\n{completion}")

Using device: cuda


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]


=== STARTING INFERENCE TESTS ===

----------------------------------------
TEST 1: PROMPT:
def quicksort(arr):
----------------------------------------
GENERATION:
def quicksort(arr):
    """Return element-wise-based indices that overlap the input array, using a
    `np.arange` operator.

    Parameters
    ----------
    arr : ndarray or Series
        Array of shape (N,) or (N, N) or other iterable with same dtype as
        ``arr``. If

----------------------------------------
TEST 2: PROMPT:
def is_palindrome(s: str) -> bool:
    """Check if a string is a palindrome."""
----------------------------------------
GENERATION:
def is_palindrome(s: str) -> bool:
    """Check if a string is a palindrome."""

    s = re.sub(" ", "_", "", s).lower()

    return not (is_palabic and s in [".", ".", 1, "+"])

----------------------------------------
TEST 3: PROMPT:
def fibonacci(n):
----------------------------------------
GENERATION:
def fibonacci(n):
    """Determines the number of times a 

In [ ]:
print(generate_text("भारत एक", max_new_tokens=40, do_sample=True))

भारत एक अंतरराष्ट्रीय क्रिकेटर है जो खुलना और गेंद से दूर रहता है।

